# 06 - COVID Impact Comparison

COVID-19 caused the largest single-year drop in global life expectancy since records began.
This notebook compares the pre-COVID (1960-2019) and full (1960-2023) datasets side by side
to quantify exactly how the pandemic distorted the Blue Zones trend analysis.

**Key questions:**
- How much did COVID set back global life expectancy?
- Were Blue Zone countries hit harder or lighter than the global average?
- Has the world recovered by 2023, or is there lasting damage?
- Did COVID change the convergence story?

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

print(f'Project directory: {PROJECT_DIR}')

BZ_ISOS = ['USA', 'JPN', 'ITA', 'GRC', 'CRI']
BZ_NAMES = {
    'USA': 'United States', 'JPN': 'Japan', 'ITA': 'Italy',
    'GRC': 'Greece', 'CRI': 'Costa Rica'
}
BZ_COLORS = {'USA': '#E74C3C', 'JPN': '#3498DB', 'ITA': '#2ECC71', 'GRC': '#9B59B6', 'CRI': '#F39C12'}

Project directory: /home/yeblad/Blue-Zones-Longevity-Analysis


## 1. Load Data

In [2]:
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
impact = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'covid_comparison', 'country_covid_impact.csv'))
summary = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'covid_comparison', 'comparison_summary.csv'))

gap_pre = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'pre_covid', 'blue_zone_vs_global.csv'))
gap_full = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'full_period', 'blue_zone_vs_global.csv'))
sigma_pre = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'pre_covid', 'sigma_convergence.csv'))
sigma_full = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'full_period', 'sigma_convergence.csv'))
beta_pre = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'pre_covid', 'beta_convergence.csv'))
beta_full = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'full_period', 'beta_convergence.csv'))

s = summary.iloc[0]
print(f'Historical data: {len(df):,} rows, {df["iso_code"].nunique()} countries')
print(f'COVID impact data: {len(impact)} countries')
print(f'Pre-COVID period: 1960-{int(gap_pre["year"].max())}')
print(f'Full period: 1960-{int(gap_full["year"].max())}')

Historical data: 5,952 rows, 93 countries
COVID impact data: 93 countries
Pre-COVID period: 1960-2019
Full period: 1960-2023


## 2. The COVID Shock: Global Life Expectancy Drop

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(gap_full['year'], gap_full['global_mean'], color='steelblue', linewidth=2, label='Global Average')
ax1.plot(gap_full['year'], gap_full['blue_zone_mean'], color='crimson', linewidth=2, label='Blue Zone Average')
ax1.axvspan(2020, 2023, alpha=0.15, color='red', label='COVID period')
ax1.set_xlabel('Year')
ax1.set_ylabel('Life Expectancy (years)')
ax1.set_title('Full Timeline: Global vs Blue Zone Average')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

zoom = gap_full[(gap_full['year'] >= 2015) & (gap_full['year'] <= 2023)]
ax2.plot(zoom['year'], zoom['global_mean'], 'o-', color='steelblue', linewidth=2.5, markersize=8, label='Global Average')
ax2.plot(zoom['year'], zoom['blue_zone_mean'], 's-', color='crimson', linewidth=2.5, markersize=8, label='Blue Zone Average')
ax2.axvspan(2020, 2021.5, alpha=0.15, color='red')
ax2.set_xlabel('Year')
ax2.set_ylabel('Life Expectancy (years)')
ax2.set_title('Zoomed: 2015-2023')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_covid_shock.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\nGlobal average drop 2019 to 2020: {s["global_drop_2020"]:.2f} years')
print(f'Global average drop 2019 to 2021: {s["global_drop_2021"]:.2f} years')


Global average drop 2019 to 2020: -0.79 years
Global average drop 2019 to 2021: -1.64 years


/tmp/ipykernel_1594406/4126757398.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Country-Level COVID Impact

In [4]:
impact_sorted = impact.sort_values('drop_2020')

fig, ax = plt.subplots(figsize=(18, 6))
colors = ['crimson' if iso in BZ_ISOS else '#95a5a6' for iso in impact_sorted['iso_code']]
bars = ax.bar(range(len(impact_sorted)), impact_sorted['drop_2020'], color=colors, width=0.8)
ax.set_xticks(range(len(impact_sorted)))
ax.set_xticklabels(impact_sorted['iso_code'], rotation=90, fontsize=7)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('LE Change 2019 to 2020 (years)')
ax.set_title('Life Expectancy Change 2019 to 2020 by Country (Red = Blue Zone)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_country_covid_impact.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Worst hit: {s["worst_hit_country"]} ({s["worst_hit_drop"]:.2f} years)')
print(f'Countries with LE drop in 2020: {(impact["drop_2020"] < 0).sum()} / {len(impact)}')

Worst hit: ECU (-5.28 years)
Countries with LE drop in 2020: 70 / 93


/tmp/ipykernel_1594406/436999659.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Blue Zone Countries: Individual COVID Impact

In [5]:
bz_impact = impact[impact['is_blue_zone'] == 1].copy()
bz_impact['country'] = bz_impact['iso_code'].map(BZ_NAMES)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(bz_impact))
width = 0.25
ax.bar(x - width, bz_impact['drop_2020'], width, label='Drop 2019-2020', color='#e74c3c', alpha=0.8)
ax.bar(x, bz_impact['drop_2021'].fillna(0), width, label='Drop 2019-2021', color='#c0392b', alpha=0.8)
recovery_colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in bz_impact['recovery_2023'].fillna(0)]
ax.bar(x + width, bz_impact['recovery_2023'].fillna(0), width, label='Net 2019-2023', color=recovery_colors, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(bz_impact['country'])
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Change in Life Expectancy (years)')
ax.set_title('Blue Zone Countries: COVID Impact and Recovery')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_bz_covid_impact.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('Blue Zone COVID impact:')
for _, row in bz_impact.sort_values('drop_2020').iterrows():
    status = 'recovered' if row['recovery_2023'] >= 0 else 'still below 2019'
    print(f'  {row["country"]}: 2020 drop = {row["drop_2020"]:+.2f}, 2023 net = {row["recovery_2023"]:+.2f} ({status})')

Blue Zone COVID impact:
  United States: 2020 drop = -1.81, 2023 net = -0.40 (still below 2019)
  Italy: 2020 drop = -1.30, 2023 net = +0.20 (recovered)
  Costa Rica: 2020 drop = -0.57, 2023 net = +0.50 (recovered)
  Greece: 2020 drop = -0.35, 2023 net = -0.10 (still below 2019)
  Japan: 2020 drop = +0.20, 2023 net = -0.32 (still below 2019)


/tmp/ipykernel_1594406/68109399.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Gap Analysis: Pre-COVID vs Full Period

In [6]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

ax1.plot(gap_pre['year'], gap_pre['bz_gap_over_global'], color='#2ecc71', linewidth=2.5, label='Pre-COVID (through 2019)')
ax1.plot(gap_full['year'], gap_full['bz_gap_over_global'], color='#e74c3c', linewidth=2.5, label='Full Period (through 2023)')
ax1.fill_between(gap_pre['year'], gap_pre['bz_gap_over_global'], alpha=0.15, color='#2ecc71')
ax1.axvspan(2020, 2023, alpha=0.1, color='red')
ax1.set_ylabel('BZ Advantage Over Global (years)')
ax1.set_title('Blue Zone Gap: Pre-COVID vs Full Period')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(sigma_pre['year'], sigma_pre['le_std'], color='#2ecc71', linewidth=2.5, label='Pre-COVID')
ax2.plot(sigma_full['year'], sigma_full['le_std'], color='#e74c3c', linewidth=2.5, label='Full Period')
ax2.axvspan(2020, 2023, alpha=0.1, color='red')
ax2.set_xlabel('Year')
ax2.set_ylabel('Standard Deviation (years)')
ax2.set_title('Sigma Convergence: Pre-COVID vs Full Period')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_gap_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

pre_gap = gap_pre.iloc[-1]['bz_gap_over_global']
full_gap = gap_full.iloc[-1]['bz_gap_over_global']
print(f'Pre-COVID final gap (2019): {pre_gap:.1f} years')
print(f'Full period final gap (2023): {full_gap:.1f} years')
print(f'Difference: {full_gap - pre_gap:.1f} years')
if full_gap < pre_gap:
    print(f'COVID accelerated convergence by {pre_gap - full_gap:.1f} years')

Pre-COVID final gap (2019): 6.4 years
Full period final gap (2023): 5.9 years
Difference: -0.5 years
COVID accelerated convergence by 0.5 years


/tmp/ipykernel_1594406/3228689592.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Beta Convergence: Pre-COVID vs Full Period

In [7]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(beta_pre['decade'], beta_pre['beta_correlation'], color='#2ecc71', alpha=0.8)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.set_ylabel('Beta Correlation')
ax1.set_title('Beta Convergence: Pre-COVID (1960-2019)')
ax1.set_ylim(-0.8, 0.2)
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(beta_pre['beta_correlation']):
    ax1.text(i, v - 0.03, f'{v:.2f}', ha='center', fontsize=9)

ax2.bar(beta_full['decade'], beta_full['beta_correlation'], color='#e74c3c', alpha=0.8)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_ylabel('Beta Correlation')
ax2.set_title('Beta Convergence: Full Period (1960-2023)')
ax2.set_ylim(-0.8, 0.2)
ax2.grid(axis='y', alpha=0.3)
for i, v in enumerate(beta_full['beta_correlation']):
    ax2.text(i, v - 0.03, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_beta_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('Pre-COVID beta convergence:')
for _, row in beta_pre.iterrows():
    print(f'  {row["decade"]}: r={row["beta_correlation"]:.3f} ({row["convergence"]})')
print()
print('Full period beta convergence:')
for _, row in beta_full.iterrows():
    print(f'  {row["decade"]}: r={row["beta_correlation"]:.3f} ({row["convergence"]})')

Pre-COVID beta convergence:
  1960s: r=-0.581 (Yes)
  1970s: r=-0.450 (Yes)
  1980s: r=-0.213 (Yes)
  1990s: r=-0.088 (Weak)
  2000s: r=-0.652 (Yes)

Full period beta convergence:
  1960s: r=-0.581 (Yes)
  1970s: r=-0.450 (Yes)
  1980s: r=-0.213 (Yes)
  1990s: r=-0.088 (Weak)
  2000s: r=-0.652 (Yes)
  2010s: r=-0.569 (Yes)


/tmp/ipykernel_1594406/2387526242.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Recovery Status by 2023

## 8. Overlay Analysis: Trend vs Reality

These charts overlay pre-COVID trend extrapolations against actual 2020-2023 data to show exactly where and how much COVID deviated from the expected trajectory.

In [8]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (label, is_bz) in zip(axes, [("Global Average", None), ("Blue Zone Average", True)]):
    if is_bz:
        subset = df[df['is_blue_zone'] == 1]
    else:
        subset = df
    yearly = subset.groupby('year')['life_expectancy'].mean().dropna()
    trend_years = yearly[(yearly.index >= 2000) & (yearly.index <= 2019)]
    slope, intercept, _, _, _ = stats.linregress(trend_years.index, trend_years.values)
    extrap_years = np.arange(2000, 2024)
    extrap_values = slope * extrap_years + intercept

    actual = yearly[yearly.index >= 2000]
    ax.plot(actual.index, actual.values, 'o-', color='#2c3e50', linewidth=2.5, markersize=6,
            label='Actual Data', zorder=5)
    ax.plot(extrap_years, extrap_values, '--', color='#27ae60', linewidth=2,
            label='Pre-COVID Trend (2000-2019)', alpha=0.8)

    for yr in [2020, 2021, 2022, 2023]:
        if yr in actual.index:
            actual_val = actual[yr]
            trend_val = slope * yr + intercept
            if actual_val < trend_val:
                ax.fill_between([yr-0.3, yr+0.3], actual_val, trend_val,
                              color='#e74c3c', alpha=0.3)
                ax.annotate(f'{actual_val - trend_val:+.1f}',
                          xy=(yr, (actual_val + trend_val)/2), fontsize=9,
                          ha='center', color='#c0392b', fontweight='bold')

    ax.axvspan(2020, 2023, alpha=0.08, color='red')
    ax.set_xlabel('Year')
    ax.set_ylabel('Life Expectancy (years)')
    ax.set_title(f'{label}: Trend vs Reality')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('COVID Deviation from Pre-Pandemic Trend', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_trend_vs_reality.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Red shading shows how far actual data fell below the pre-COVID trend.')

Red shading shows how far actual data fell below the pre-COVID trend.


/tmp/ipykernel_1594406/1470900137.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. All Blue Zone Countries Overlaid: COVID Years Zoomed

In [9]:
fig, ax = plt.subplots(figsize=(14, 7))

for iso in BZ_ISOS:
    c = df[df['iso_code'] == iso].sort_values('year')
    zoom = c[(c['year'] >= 2015) & (c['year'] <= 2023)][['year', 'life_expectancy']].dropna()
    ax.plot(zoom['year'], zoom['life_expectancy'], 'o-', color=BZ_COLORS[iso],
            linewidth=2.5, markersize=8, label=BZ_NAMES[iso])
    for yr in [2019, 2020]:
        val = zoom[zoom['year'] == yr]['life_expectancy']
        if len(val):
            ax.annotate(f'{val.iloc[0]:.1f}', xy=(yr, val.iloc[0]),
                       textcoords="offset points", xytext=(0, 10 if yr == 2019 else -15),
                       fontsize=8, ha='center', color=BZ_COLORS[iso])

global_avg = df.groupby('year')['life_expectancy'].mean()
zoom_global = global_avg[(global_avg.index >= 2015) & (global_avg.index <= 2023)]
ax.plot(zoom_global.index, zoom_global.values, 's--', color='#95a5a6', linewidth=2,
        markersize=6, label='Global Average')

ax.axvspan(2020, 2021.5, alpha=0.12, color='red', label='COVID peak impact')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Life Expectancy (years)', fontsize=12)
ax.set_title('Blue Zone Countries: COVID Years Zoomed (2015-2023)', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(2015, 2024))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_bz_overlay_zoomed.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Japan barely dipped. The US cratered. The spread tells the COVID story.')

Japan barely dipped. The US cratered. The spread tells the COVID story.


/tmp/ipykernel_1594406/472054005.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Recovery Trajectory: Change Relative to 2019 Baseline

In [10]:
fig, ax = plt.subplots(figsize=(12, 7))

for iso in BZ_ISOS:
    c = df[df['iso_code'] == iso].sort_values('year')
    le_2019 = c[c['year'] == 2019]['life_expectancy'].iloc[0]
    years_data = c[(c['year'] >= 2019) & (c['year'] <= 2023)][['year', 'life_expectancy']].dropna()
    if not years_data.empty:
        relative = years_data['life_expectancy'].values - le_2019
        ax.plot(years_data['year'], relative, 'o-', color=BZ_COLORS[iso],
                linewidth=2.5, markersize=8, label=BZ_NAMES[iso])
        ax.annotate(f'{relative[-1]:+.1f}', xy=(2023, relative[-1]),
                   textcoords="offset points", xytext=(8, 0),
                   fontsize=10, color=BZ_COLORS[iso], fontweight='bold')

# Global average
global_vals = []
global_yrs = []
global_2019 = df[df['year'] == 2019]['life_expectancy'].mean()
for yr in range(2019, 2024):
    g = df[df['year'] == yr]['life_expectancy'].mean()
    global_vals.append(g - global_2019)
    global_yrs.append(yr)
ax.plot(global_yrs, global_vals, 's--', color='#95a5a6', linewidth=2, markersize=6, label='Global Average')

ax.axhline(y=0, color='black', linewidth=1)
ax.fill_between([2019, 2023], -3, 0, alpha=0.05, color='red')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Change from 2019 Level (years)', fontsize=12)
ax.set_title('COVID Recovery Trajectory: Change Relative to 2019 Baseline', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(2019, 2024))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_recovery_trajectory.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Below zero = still worse than 2019. The US never made it back.')

Below zero = still worse than 2019. The US never made it back.


/tmp/ipykernel_1594406/1432914159.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Convergence Gap: COVID Disruption Overlay

In [11]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: BZ gap with trend overlay
ax1.plot(gap_full['year'], gap_full['bz_gap_over_global'], '-', color='#2c3e50',
        linewidth=2.5, label='Actual BZ Gap')
ax1.fill_between(gap_full['year'], gap_full['bz_gap_over_global'], alpha=0.15, color='#3498db')

gap_trend = gap_full[(gap_full['year'] >= 1990) & (gap_full['year'] <= 2019)]
slope, intercept, _, _, _ = stats.linregress(gap_trend['year'], gap_trend['bz_gap_over_global'])
extrap_years = np.arange(1990, 2024)
ax1.plot(extrap_years, slope * extrap_years + intercept, '--', color='#27ae60',
        linewidth=2, label='Pre-COVID Convergence Trend', alpha=0.8)
ax1.axvspan(2020, 2023, alpha=0.1, color='red', label='COVID period')

for yr in [2019, 2021, 2023]:
    row = gap_full[gap_full['year'] == yr]
    if len(row):
        val = row['bz_gap_over_global'].iloc[0]
        ax1.annotate(f'{val:.1f}', xy=(yr, val), textcoords="offset points",
                   xytext=(0, 10), fontsize=10, ha='center', fontweight='bold')

ax1.set_xlabel('Year')
ax1.set_ylabel('Blue Zone Gap (years)')
ax1.set_title('BZ Advantage: COVID Disruption')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: Sigma with COVID spike
ax2.plot(sigma_full['year'], sigma_full['le_std'], '-', color='#8e44ad', linewidth=2.5)
ax2.fill_between(sigma_full['year'], sigma_full['le_std'], alpha=0.12, color='#8e44ad')

covid_sigma = sigma_full[(sigma_full['year'] >= 2020) & (sigma_full['year'] <= 2021)]
ax2.plot(covid_sigma['year'], covid_sigma['le_std'], 'o', color='#e74c3c', markersize=10, zorder=5)

s_trend = sigma_full[(sigma_full['year'] >= 1990) & (sigma_full['year'] <= 2019)]
slope2, intercept2, _, _, _ = stats.linregress(s_trend['year'], s_trend['le_std'])
ax2.plot(np.arange(1990, 2024), slope2 * np.arange(1990, 2024) + intercept2,
        '--', color='#27ae60', linewidth=2, alpha=0.7, label='Pre-COVID Trend')
ax2.axvspan(2020, 2023, alpha=0.08, color='red')

ax2.set_xlabel('Year')
ax2.set_ylabel('Standard Deviation (years)')
ax2.set_title('Global LE Spread: COVID Spike')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_convergence_covid_overlay.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('The gap spiked in 2021 before dropping below the pre-COVID trend by 2023.')

The gap spiked in 2021 before dropping below the pre-COVID trend by 2023.


/tmp/ipykernel_1594406/2098329409.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(18, 6))

recovery = impact.dropna(subset=['recovery_2023']).sort_values('recovery_2023')
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in recovery['recovery_2023']]
bars = ax.bar(range(len(recovery)), recovery['recovery_2023'], color=colors, width=0.8)

for i, iso in enumerate(recovery['iso_code']):
    if iso in BZ_ISOS:
        bars[i].set_edgecolor('black')
        bars[i].set_linewidth(2)

ax.set_xticks(range(len(recovery)))
ax.set_xticklabels(recovery['iso_code'], rotation=90, fontsize=7)
ax.axhline(y=0, color='black', linewidth=1)
ax.set_ylabel('Net LE Change 2019 to 2023 (years)')
ax.set_title('Recovery Status by 2023: Green = Recovered, Red = Still Below 2019 (BZ countries outlined)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb06_recovery_status.png'),
            dpi=150, bbox_inches='tight')
plt.show()

n_recovered = (recovery['recovery_2023'] >= 0).sum()
print(f'Countries recovered by 2023: {n_recovered} / {len(recovery)}')
print(f'Countries still below 2019 levels: {len(recovery) - n_recovered}')

Countries recovered by 2023: 75 / 93
Countries still below 2019 levels: 18


/tmp/ipykernel_1594406/1102240101.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Summary

In [13]:
print('COVID IMPACT COMPARISON SUMMARY')
print('=' * 50)
print()
print(f'Global LE drop 2019-2020: {s["global_drop_2020"]:.2f} years')
print(f'Global LE drop 2019-2021: {s["global_drop_2021"]:.2f} years')
print(f'Worst hit country: {s["worst_hit_country"]} ({s["worst_hit_drop"]:.2f} years)')
print()
print(f'BZ avg 2020 drop: {s["avg_drop_bz"]:.2f} years')
print(f'Non-BZ avg 2020 drop: {s["avg_drop_non_bz"]:.2f} years')
print()
print(f'Pre-COVID gap (2019): {s["pre_covid_gap"]:.1f} years')
print(f'Full period gap (2023): {s["full_2023_gap"]:.1f} years')
gap_change = s['full_2023_gap'] - s['pre_covid_gap']
if gap_change < 0:
    print(f'COVID accelerated convergence by {abs(gap_change):.1f} years')
else:
    print(f'COVID slowed convergence by {gap_change:.1f} years')
print()
print(f'Countries recovered by 2023: {int(s["n_countries_recovered"])} / 93')
print()
print('KEY FINDING: The pre-COVID dataset (1960-2019) provides the cleanest')
print('picture of secular trends. The full dataset (1960-2023) is more honest')
print('but includes a massive shock. Both are valid -- they answer different questions.')

COVID IMPACT COMPARISON SUMMARY

Global LE drop 2019-2020: -0.79 years
Global LE drop 2019-2021: -1.64 years
Worst hit country: ECU (-5.28 years)

BZ avg 2020 drop: -0.77 years
Non-BZ avg 2020 drop: -0.79 years

Pre-COVID gap (2019): 6.4 years
Full period gap (2023): 5.9 years
COVID accelerated convergence by 0.5 years

Countries recovered by 2023: 75 / 93

KEY FINDING: The pre-COVID dataset (1960-2019) provides the cleanest
picture of secular trends. The full dataset (1960-2023) is more honest
but includes a massive shock. Both are valid -- they answer different questions.


## 13. Structural Break and Time-Series Tests

Statistical tests confirming that COVID-19 caused a structural break in life expectancy trends.

In [14]:
# Load time-series test results
STAT_DIR = os.path.join(PROJECT_DIR, 'outputs', 'analysis')
ts_tests = pd.read_csv(os.path.join(STAT_DIR, 'time_series_tests.csv'))
covid_accel = pd.read_csv(os.path.join(STAT_DIR, 'covid_acceleration_test.csv'))

print("TIME-SERIES STATISTICAL TESTS")
print("=" * 60)
for _, row in ts_tests.iterrows():
    print(f"\n{row['test']}:")
    print(f"  {row['description']}")
    print(f"  Statistic: {row['statistic']:.3f}, p-value: {row['p_value']:.4f}")
    print(f"  Conclusion: {row['conclusion']}")

print("\n\nCOVID ACCELERATION TEST")
print("=" * 60)
print("Did the 2020-2023 BZ gap fall outside the pre-COVID trend prediction interval?")
print()
for _, row in covid_accel.iterrows():
    flag = " *** OUTSIDE PI" if row['outside_pi'] else ""
    print(f"  {int(row['year'])}: predicted gap={row['predicted_gap']:.2f}, "
          f"actual={row['actual_gap']:.2f}, diff={row['difference']:.2f}{flag}")

print("\nKey finding: Structural break at 2020 is confirmed (Chow test p<0.001).")
print("The 2022 gap fell outside the pre-COVID trend prediction interval,")
print("indicating COVID accelerated convergence beyond expected levels.")

TIME-SERIES STATISTICAL TESTS

ADF_global_LE:
  Augmented Dickey-Fuller test on global average LE
  Statistic: -2.820, p-value: 0.0555
  Conclusion: Non-stationary (trending)

ADF_global_LE_diff:
  ADF on first differences of global LE
  Statistic: -4.101, p-value: 0.0010
  Conclusion: Stationary after differencing

ADF_BZ_gap:
  ADF test on BZ-global gap series
  Statistic: 2.111, p-value: 0.9988
  Conclusion: Non-stationary

Chow_structural_break_2020:
  Structural break test at 2020 on BZ gap series
  Statistic: 10.824, p-value: 0.0001
  Conclusion: Structural break detected

ARIMA_COVID_shock:
  ARIMA(1,1,0) residual at 2020 as z-score
  Statistic: 0.013, p-value: 0.9898
  Conclusion: Within normal variation


COVID ACCELERATION TEST
Did the 2020-2023 BZ gap fall outside the pre-COVID trend prediction interval?

  2020: predicted gap=6.98, actual=6.77, diff=-0.21
  2021: predicted gap=6.88, actual=6.99, diff=0.11
  2022: predicted gap=6.78, actual=5.95, diff=-0.84 *** OUTSIDE PI
  